# 99 · Fusión de Grafos Multidimensional

Fusiona los tres grafos del pipeline en un único grafo enriquecido:
- **Dimensiones**: transcripción, año, vestimenta
- **Cross-links**: foto↔foto (vestimenta/palabras), palabra→año, vestimenta→año
- **Análisis**: comunidades Louvain, betweenness, degree, clustering
- **Exportación**: GEXF + nodos.csv + aristas.csv

> Para añadir dimensiones nuevas: genera `output/DIM/grafo_DIM.gexf` y añade la entrada a `GEXF_SOURCES`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [12]:
import os
import pandas as pd
import networkx as nx
from collections import Counter
from itertools import combinations

BASE = '/content/drive/MyDrive/TFM-Sara'

# Registro de dimensiones — añade entradas aquí para nuevas dimensiones
GEXF_SOURCES = {
    'transcripcion': f'{BASE}/output/transcripciones/knowledge_graph_palabras.gexf',
    'anyo':          f'{BASE}/output/predicciones_anyo/grafo_anyos.gexf',
    'vestimenta':    f'{BASE}/output/vestimenta/grafo_vestimenta.gexf',
    # 'postura':     f'{BASE}/output/postura/grafo_postura.gexf',
}
SUPPORT_CSV = f'{BASE}/support_data/input.csv'
OUTPUT_DIR  = f'{BASE}/output/grafo_final'
OUTPUT_GEXF = f'{OUTPUT_DIR}/grafo_multidimensional.gexf'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'NetworkX {nx.__version__}  |  Rutas OK')

NetworkX 3.6.1  |  Rutas OK


In [14]:
from networkx.readwrite.gexf import GEXFReader

def _tolerant_int(val):
    """Convierte int tolerantemente: acepta strings float como '13.33...'."""
    try:
        return int(val)
    except ValueError:
        return int(float(val))

# python_type es un atributo de INSTANCIA (no de clase), por eso
# hay que parchearlo dentro de __init__ tras la inicializacion base.
_orig_gexf_init = GEXFReader.__init__

def _patched_gexf_init(self, *args, **kwargs):
    _orig_gexf_init(self, *args, **kwargs)
    self.python_type['integer'] = _tolerant_int

GEXFReader.__init__ = _patched_gexf_init
print('GEXF reader parcheado (int tolerante OK)')


def load_and_merge(sources: dict) -> nx.Graph:
    """Carga y fusiona múltiples GEXF. Nodos con mismo ID tienen atributos combinados."""
    G = nx.Graph()
    for dim_name, path in sources.items():
        if not os.path.exists(path):
            print(f'  WARN [{dim_name}] no encontrado: {path}')
            continue
        sub = nx.read_gexf(path)
        for node, data in sub.nodes(data=True):
            if G.has_node(node):
                G.nodes[node].update(data)
            else:
                G.add_node(node, **data)
        for u, v, data in sub.edges(data=True):
            if not G.has_edge(u, v):
                G.add_edge(u, v, **data)
        print(f'  OK  [{dim_name:15s}] {sub.number_of_nodes():5d} nodos  {sub.number_of_edges():5d} aristas')
    return G

print('Cargando grafos fuente...')
full_graph = load_and_merge(GEXF_SOURCES)

dim_counts = Counter(d.get('dimension', '?') for _, d in full_graph.nodes(data=True))
print(f'\nGrafo fusionado: {full_graph.number_of_nodes()} nodos, {full_graph.number_of_edges()} aristas')
print('Nodos por dimension:')
for dim, cnt in sorted(dim_counts.items(), key=lambda x: -x[1]):
    print(f'  {dim:25s}: {cnt}')

GEXF reader parcheado (int tolerante OK)
Cargando grafos fuente...


RecursionError: maximum recursion depth exceeded

In [ ]:
# Limpiar espacios en PKs de nodos imagen
renames = {n: n.strip() for n, d in full_graph.nodes(data=True)
           if d.get('dimension') == 'imagen' and n != n.strip()}
if renames:
    nx.relabel_nodes(full_graph, renames, copy=False)
    print(f'  Renombrados {len(renames)} nodos imagen (espacios eliminados)')

# Atributos visuales consistentes en nodos imagen
for node, data in full_graph.nodes(data=True):
    if data.get('dimension') == 'imagen' or data.get('group') == 1:
        full_graph.nodes[node].update({
            'dimension': 'imagen',
            'color':     '#4A90D9',
            'group':     1,
            'size':      data.get('size', 25.0),
            'pk':        node,
        })

foto_nodes = [n for n, d in full_graph.nodes(data=True) if d.get('dimension') == 'imagen']
print(f'OK Nodos imagen normalizados: {len(foto_nodes)}')

# Enriquecer nodos foto con metadatos del CSV de soporte
ENRICH_COLS = ['caption', 'nom_fons', 'toponims', 'noms_propis', 'year']
if os.path.exists(SUPPORT_CSV):
    df_sup = pd.read_csv(SUPPORT_CSV)
    lookup = {
        str(row.get('filename', '')).strip(): {
            c: row[c] for c in ENRICH_COLS if c in row and pd.notna(row[c])
        }
        for _, row in df_sup.iterrows()
    }
    n_enr = 0
    for node in foto_nodes:
        if node in lookup:
            full_graph.nodes[node].update(lookup[node])
            n_enr += 1
    print(f'OK Enriquecidos con CSV: {n_enr} / {len(foto_nodes)} fotos')
else:
    print(f'  WARN {SUPPORT_CSV} no encontrado (enriquecimiento omitido)')

In [ ]:
# Indices foto -> {vestimenta, palabras, anyos}
foto_vest  = {}
foto_words = {}
foto_years = {}

for u, v, data in full_graph.edges(data=True):
    rel   = data.get('relation', '')
    dim_u = full_graph.nodes[u].get('dimension', '')
    foto  = u if dim_u == 'imagen' else v
    other = v if dim_u == 'imagen' else u
    if rel == 'lleva_puesto':
        foto_vest.setdefault(foto, {})[other] = max(
            foto_vest.get(foto, {}).get(other, 0.0), data.get('weight', 1.0))
    elif rel == 'contiene_palabra':
        foto_words.setdefault(foto, set()).add(other)
    elif full_graph.nodes[other].get('dimension') == 'anyo':
        foto_years.setdefault(foto, set()).add(other)

print(f'Indices | vestimenta: {len(foto_vest)}  palabras: {len(foto_words)}  anyos: {len(foto_years)}')

# 1. Foto <-> Foto por vestimenta compartida
n_vest_sim = 0
for fa, fb in combinations(foto_vest.keys(), 2):
    shared = set(foto_vest[fa]) & set(foto_vest[fb])
    if shared and full_graph.has_node(fa) and full_graph.has_node(fb):
        w = sum(min(foto_vest[fa][p], foto_vest[fb][p]) for p in shared)
        if full_graph.has_edge(fa, fb):
            full_graph[fa][fb]['weight'] = full_graph[fa][fb].get('weight', 0) + w
        else:
            full_graph.add_edge(fa, fb, relation='comparte_vestimenta',
                                weight=round(w, 4), shared_items=len(shared),
                                dimension='similitud_vestimenta')
            n_vest_sim += 1
print(f'OK Foto<->Foto vestimenta compartida : {n_vest_sim}')

# 2. Foto <-> Foto por palabras compartidas (Jaccard >= 0.10)
MIN_JACCARD = 0.10
n_word_sim  = 0
for fa, fb in combinations(foto_words.keys(), 2):
    sa, sb  = foto_words[fa], foto_words[fb]
    union   = sa | sb
    jaccard = len(sa & sb) / len(union) if union else 0
    if jaccard >= MIN_JACCARD and full_graph.has_node(fa) and full_graph.has_node(fb):
        if full_graph.has_edge(fa, fb):
            full_graph[fa][fb]['weight'] = full_graph[fa][fb].get('weight', 0) + jaccard
        else:
            full_graph.add_edge(fa, fb, relation='comparte_palabras',
                                weight=round(jaccard, 4), dimension='similitud_textual')
            n_word_sim += 1
print(f'OK Foto<->Foto palabras compartidas  : {n_word_sim}')

# 3. Palabra -> Anyo
yr_word_cnt = {}
for foto, words in foto_words.items():
    for yr in foto_years.get(foto, set()):
        for w in words:
            yr_word_cnt[(w, yr)] = yr_word_cnt.get((w, yr), 0) + 1
n_pw = 0
for (word, year), cnt in yr_word_cnt.items():
    if not full_graph.has_edge(word, year):
        full_graph.add_edge(word, year, relation='aparece_en_anyo',
                            weight=float(cnt), dimension='transcripcion_anyo')
        n_pw += 1
print(f'OK Palabra -> Anyo                   : {n_pw}')

# 4. Vestimenta -> Anyo
yr_vest_cnt = {}
for foto, vests in foto_vest.items():
    for yr in foto_years.get(foto, set()):
        for v in vests:
            yr_vest_cnt[(v, yr)] = yr_vest_cnt.get((v, yr), 0) + 1
n_vy = 0
for (vest, year), cnt in yr_vest_cnt.items():
    if not full_graph.has_edge(vest, year):
        full_graph.add_edge(vest, year, relation='vestimenta_en_anyo',
                            weight=float(cnt), dimension='vestimenta_anyo')
        n_vy += 1
print(f'OK Vestimenta -> Anyo                : {n_vy}')
print(f'\nTotal aristas: {full_graph.number_of_edges()}')

In [ ]:
COMMUNITY_PALETTE = [
    '#E63946','#457B9D','#2A9D8F','#E9C46A','#F4A261',
    '#8338EC','#3A86FF','#FB5607','#06D6A0','#118AB2',
    '#FFD166','#073B4C','#EF476F','#A8DADC','#6D6875',
    '#B5838D','#E07A5F','#3D405B','#81B29A','#F2CC8F',
]

# Louvain nativo de NetworkX (>= 2.7); fallback a python-louvain
try:
    from networkx.algorithms.community import louvain_communities
    comm_sets = louvain_communities(full_graph, seed=42)
    partition = {node: cid for cid, members in enumerate(comm_sets) for node in members}
    print(f'OK Louvain (networkx built-in): {len(comm_sets)} comunidades')
except (ImportError, AttributeError):
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-louvain', '-q'])
    import community.community_louvain as _cl
    partition = _cl.best_partition(full_graph, random_state=42)
    print(f'OK Louvain (python-louvain): {len(set(partition.values()))} comunidades')

for node, cid in partition.items():
    full_graph.nodes[node]['community']       = cid
    full_graph.nodes[node]['community_color'] = COMMUNITY_PALETTE[cid % len(COMMUNITY_PALETTE)]

comm_sizes = Counter(partition.values())
print('Top 5 comunidades por tamano:')
for cid, size in comm_sizes.most_common(5):
    print(f'  Comunidad {cid:3d}: {size:4d} nodos  {COMMUNITY_PALETTE[cid % len(COMMUNITY_PALETTE)]}')

print('\nCalculando metricas de centralidad...')
deg_c  = nx.degree_centrality(full_graph)
betw_c = nx.betweenness_centrality(full_graph, normalized=True)
clust  = nx.clustering(full_graph)

for node in full_graph.nodes():
    full_graph.nodes[node]['degree']         = full_graph.degree(node)
    full_graph.nodes[node]['deg_centrality'] = round(deg_c[node], 6)
    full_graph.nodes[node]['betweenness']    = round(betw_c[node], 6)
    full_graph.nodes[node]['clustering']     = round(clust[node], 6)

print('Top 5 nodos por betweenness centrality:')
for nid, val in sorted(betw_c.items(), key=lambda x: -x[1])[:5]:
    dim = full_graph.nodes[nid].get('dimension', '?')
    print(f'  {nid[:50]:50s} [{dim:20s}] {val:.4f}')

In [ ]:
sep = '=' * 62
print(sep)
print('GRAFO MULTIDIMENSIONAL - RESUMEN FINAL')
print(sep)
print(f'  Nodos totales    : {full_graph.number_of_nodes():>6}')
print(f'  Aristas totales  : {full_graph.number_of_edges():>6}')
print(f'  Comunidades      : {len(set(partition.values())):>6}')
try:
    print(f'  Densidad         : {nx.density(full_graph):>12.6f}')
    print(f'  Clustering medio : {nx.average_clustering(full_graph):>12.6f}')
except Exception:
    pass

dim_counts = Counter(d.get('dimension', 'desconocida') for _, d in full_graph.nodes(data=True))
print('\nNodos por dimension:')
for dim, cnt in sorted(dim_counts.items(), key=lambda x: -x[1]):
    bar = chr(9608) * min(cnt, 40)
    print(f'  {dim:25s}: {cnt:5d}  {bar}')

rel_counts = Counter(d.get('relation', 'sin_tipo') for _, _, d in full_graph.edges(data=True))
print('\nAristas por tipo de relacion:')
for rel, cnt in sorted(rel_counts.items(), key=lambda x: -x[1]):
    print(f'  {rel:35s}: {cnt:5d}')
print(sep)

In [ ]:
nx.write_gexf(full_graph, OUTPUT_GEXF)
print(f'OK GEXF        -> {OUTPUT_GEXF}')

rows_n = [{'id': nid, **data} for nid, data in full_graph.nodes(data=True)]
rows_e = [{'source': u, 'target': v, **data} for u, v, data in full_graph.edges(data=True)]

pd.DataFrame(rows_n).to_csv(f'{OUTPUT_DIR}/nodos.csv',   index=False, encoding='utf-8')
pd.DataFrame(rows_e).to_csv(f'{OUTPUT_DIR}/aristas.csv', index=False, encoding='utf-8')

print(f'OK nodos.csv   -> {len(rows_n)} filas')
print(f'OK aristas.csv -> {len(rows_e)} filas')
print(f'Archivos en: {OUTPUT_DIR}')